In [ ]:
# Run on Google Colab
from google.cloud.bigquery import dataset

import requests, argparse
import pandas as pd
from google.colab import auth, data_table
from google.cloud import bigquery
from pandas_gbq import to_gbq
from datetime import datetime

def extract_data(data_name : str):
    url = f"https://fsbproject.vercel.app/{data_name}"
    response = requests.get(url)
    data = response.json()
    data_extracted = pd.DataFrame(data)

    data_extracted['created_date'] = data_extracted['created_date'].astype('datetime64[ns]')
    data_extracted['updated_date'] = data_extracted['updated_date'].astype('datetime64[ns]')
    data_extracted['landing_date'] = datetime.now()

    return(data_extracted)

def connect_to_gbq(id_project : str):
    auth.authenticate_user()
    client = bigquery.Client(project = id_project)
    return(client)

def generate_bq_schema(data : pd.DataFrame):
    dtype_mapping = {
    'object': 'STRING',
    'string': 'STRING',
    'Int64': 'INT64',
    'int64': 'INT64',
    'float64': 'FLOAT64',
    'boolean': 'BOOL',
    'bool': 'BOOL',
    'date': 'DATE',
    'datetime64[ns]': 'TIMESTAMP'
    }

    schema = []
    for col, dtype in data.dtypes.items():
        dtype_str = str(dtype)
        bq_type = dtype_mapping.get(dtype_str, 'STRING')
        schema.append({'name': col, 'type': bq_type})
    return(schema)

def load_to_gbq(
    data : pd.DataFrame,
    dataset_name : str,
    destination_table_name : str,
    id_project : str
):
    table_schema_datamart = generate_bq_schema(data)
    to_gbq(
        data,
        destination_table = f'{dataset_name}.{destination_table_name}',
        project_id = id_project,
        if_exists = 'append',
        table_schema = table_schema_datamart
    )
    print(f'{destination_table_name} berhasil di-load!')


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_name", type = str, help = "Data Extracted")
    parser.add_argument("--dataset_name", type = str, help = "Destination Dataset Name")
    parser.add_argument("--dest_table_name", type = str, help = "Destination Table Name")
    args = parser.parse_args()

    id_project = 'de-fsb-2026'
    data_name = args.data_name
    dataset_name = args.dataset_name
    dest_table_name = args.dest_table_name

    # Extract Data
    data_extracted = extract_data(data_name)

    # Connect to GBQ
    client = connect_to_gbq(id_project)

    # Load to GBQ
    load_to_gbq(
        data = data_extracted,
        dataset_name = dataset_name,
        destination_table_name = dest_table_name,
        id_project = id_project
        )
